# OpenMuse & Google Gemini Diagnostic & Connection Notebook

Welcome to the **OpenMuse & Gemini Setup & Connection Notebook**.

### What is OpenMuse?
OpenMuse is an open-source personal agent featuring an agent computer, browser automation, persistent memory, and task execution engine. Built using:
- **CopilotKit React Native** for the web and mobile user interfaces (`http://localhost:8081`)
- **Hono Node.js API** backend (`http://127.0.0.1:8787`)
- **TanStack AI / Google Gemini** adapter for open-ended reasoning and tool calling
- **Pydantic AI** for structured Python micro-agents and data validation

This notebook verifies your **Google Gemini API Free Tier** configuration, tests connectivity, runs a sample Pydantic AI agent, and checks all OpenMuse services.

## 1. Environment & Configuration Check
Loads `.env` from the OpenMuse project root and audits your configured keys.

In [ ]:
import os
import sys
import json
from pathlib import Path

# Ensure UTF-8 output
if sys.platform == 'win32':
    sys.stdout.reconfigure(encoding='utf-8')

root_dir = Path.cwd()
if root_dir.name != 'openmuse' and (root_dir / 'openmuse').exists():
    root_dir = root_dir / 'openmuse'

env_file = root_dir / '.env'
print(f"Project directory: {root_dir}")
print(f"Reading .env from: {env_file}")

env_vars = {}
if env_file.exists():
    with open(env_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                k, v = line.split('=', 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                env_vars[k] = v
                os.environ[k] = v
    print("✅ .env successfully loaded.")
else:
    print("❌ .env not found! Please ensure .env exists in openmuse.")

print(f"Current MODEL setting  : {env_vars.get('MODEL')}")
print(f"AGENT_BACKEND          : {env_vars.get('AGENT_BACKEND')}")
print(f"CPK_INTELLIGENCE_API_KEY: {'Configured ✅' if env_vars.get('CPK_INTELLIGENCE_API_KEY') else 'Missing ❌'}")
print(f"GEMINI_API_KEY         : {'Configured ✅' if env_vars.get('GEMINI_API_KEY') else 'Missing ❌'}")
print(f"GOOGLE_API_KEY         : {'Configured ✅' if env_vars.get('GOOGLE_API_KEY') else 'Missing ❌'}")

## 2. Test Google Gemini API Keys (Free Tier)
Tests reachability of Google's Generative Language API and queries available Gemini models.

In [ ]:
import urllib.request
import urllib.error

keys_to_test = []
if env_vars.get('GEMINI_API_KEY'):
    keys_to_test.append(env_vars['GEMINI_API_KEY'])
if env_vars.get('GEMINI_API_KEYS'):
    for k in env_vars['GEMINI_API_KEYS'].split(','):
        k = k.strip()
        if k and k not in keys_to_test:
            keys_to_test.append(k)

print(f"Found {len(keys_to_test)} Gemini key(s). Testing connectivity...")
valid_keys = []
for idx, key in enumerate(keys_to_test, 1):
    url = f"https://generativelanguage.googleapis.com/v1beta/models?key={key}"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "OpenMuse-Test/1.0"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode())
            models = [m['name'].replace('models/', '') for m in data.get('models', []) if 'gemini' in m.get('name', '')]
            masked = key[:6] + '...' + key[-4:]
            print(f"  Key #{idx} ({masked}): ✅ Valid! Found {len(models)} Gemini models.")
            valid_keys.append(key)
    except Exception as e:
        print(f"  Key #{idx}: ❌ Error: {e}")

## 3. Live Generation Test on Gemini 2.5 Flash / Flash-Lite
Sends a test prompt directly to Google's API to confirm active generation.

In [ ]:
if valid_keys:
    active_key = valid_keys[0]
    for test_model in ['gemini-2.5-flash-lite', 'gemini-2.5-flash']:
        gen_url = f"https://generativelanguage.googleapis.com/v1beta/models/{test_model}:generateContent?key={active_key}"
        payload = json.dumps({
            "contents": [{"parts": [{"text": "Confirm in 1 sentence that you are ready to power OpenMuse personal agents."}]}]
        }).encode('utf-8')
        try:
            req = urllib.request.Request(gen_url, data=payload, headers={"Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=15) as resp:
                res = json.loads(resp.read().decode())
                answer = res['candidates'][0]['content']['parts'][0]['text'].strip()
                print(f"[{test_model}] Response: {answer}")
                break
        except Exception as e:
            print(f"[{test_model}] Notice: {e} (trying next model)")
else:
    print("No valid Gemini keys to test generation.")

## 4. Run a Pydantic AI Agent with Google Gemini
Demonstrates how Pydantic AI structured outputs and typed reasoning work seamlessly with Gemini.

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

# Schema for structured agent decision
class AgentDecision(BaseModel):
    action: str = Field(description="The recommended immediate action for OpenMuse")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0")
    explanation: str = Field(description="Brief justification for this decision")
    delegated_task_needed: bool = Field(description="Whether this requires an asynchronous background task")

os.environ['PYDANTIC_AI_NO_BANNER'] = '1'
os.environ['GEMINI_API_KEY'] = valid_keys[0] if valid_keys else ''
os.environ['GOOGLE_API_KEY'] = valid_keys[0] if valid_keys else ''

print("Initializing Pydantic AI agent...")
agent = Agent(
    'google:gemini-2.5-flash-lite',
    output_type=AgentDecision,
    system_prompt=(
        "You are OpenMuse's strategic planning agent. Analyze user requests, determine whether "
        "they require instant feedback or delegated durable background tasks, and return typed decisions."
    )
)

sample_user_prompt = "Please monitor the price of flights to Tokyo and alert me if it drops below $700."
print(f"Prompt: '{sample_user_prompt}'\n")
result = agent.run_sync(sample_user_prompt)

print("✅ Structured Output from Pydantic AI:")
print(f"  • Action                : {result.output.action}")
print(f"  • Confidence            : {result.output.confidence}")
print(f"  • Explanation           : {result.output.explanation}")
print(f"  • Delegated Task Needed : {result.output.delegated_task_needed}")

## 5. Check OpenMuse Local Services Health
Tests whether the OpenMuse API (Hono), Web Frontend (Expo), and Browser Worker (Docker) are active.

In [ ]:
def check_service(name, url):
    try:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=3) as resp:
            data = resp.read().decode()
            try:
                js = json.loads(data)
                print(f"✅ {name} ({url}) is ONLINE!")
                print(f"   Payload: {json.dumps(js, indent=2)}")
            except Exception:
                print(f"✅ {name} ({url}) is ONLINE!")
    except Exception:
        print(f"⏸️  {name} ({url}) is OFFLINE.")

api_port = env_vars.get('PORT', '8787')
check_service("OpenMuse Backend API", f"http://127.0.0.1:{api_port}/api/health")
check_service("OpenMuse Web Frontend", "http://localhost:8081")
check_service("Browser Worker", f"{env_vars.get('BROWSER_WORKER_URL', 'http://127.0.0.1:8790')}/health")
check_service("AG-UI & A2UI External Agent Bridge", "http://127.0.0.1:8000/health")

## Next Steps & First Demo Experiments

To start OpenMuse and begin your first experiment:
1. Open **Terminal 1** in `openmuse` and run: `pnpm dev`
2. Open **Terminal 2** in `openmuse` and run: `pnpm dev:web`
3. Open your browser at **http://localhost:8081**

### First Demo Prompts to Try:
- **"Complete the permission slip"**: Fills out school consent form via PDF automation.
- **"Check out Hacker News for cool stuff"**: Opens the agent browser, browses stories, and summarizes them.
- **"Keep an eye on example.com"**: Sets up a background condition watch in Goals & Tracking.
- **"Organize my spending"**: Navigates to Finance and generates an interactive spending tracker.